In [ ]:
# Parameters
molecule_name = "H2"
max_iterations = 30


In [ ]:
%matplotlib inline
import matplotlib
matplotlib.use('agg')
import warnings
warnings.filterwarnings('ignore')


In [ ]:

from qiskit import QuantumCircuit, transpile
from qiskit_aer import Aer
from qiskit.visualization import circuit_drawer, plot_histogram, plot_bloch_multivector
from qiskit.quantum_info import Statevector, partial_trace, SparsePauliOp
import matplotlib.pyplot as plt
import numpy as np
import scipy.optimize as opt

mol = str(molecule_name)
max_iter = int(max_iterations)

print(f"VQE Simulation — Molecule: {mol}")

# Mock Hamiltonian for H2 (simplified for demo)
hamiltonian = SparsePauliOp.from_list([
    ("II", -1.052),
    ("IZ", 0.397),
    ("ZI", -0.397),
    ("ZZ", -0.011),
    ("XX", 0.180)
])

# Ansatz
def get_ansatz(params):
    qc = QuantumCircuit(2)
    qc.ry(params[0], 0)
    qc.ry(params[1], 1)
    qc.cx(0, 1)
    qc.ry(params[2], 0)
    qc.ry(params[3], 1)
    return qc

simulator = Aer.get_backend('statevector_simulator')
energy_history = []

def cost_function(params):
    qc = get_ansatz(params)
    compiled = transpile(qc, simulator)
    job = simulator.run(compiled)
    sv = job.result().get_statevector()
    
    # Calculate expectation value <psi|H|psi>
    expectation = sv.expectation_value(hamiltonian).real
    energy_history.append(expectation)
    return expectation

initial_params = np.zeros(4)
result = opt.minimize(cost_function, initial_params, method='COBYLA', options={'maxiter': max_iter})

print(f"Optimized Ground State Energy: {result.fun:.4f} Hartree")

# Plot convergence
plt.figure(figsize=(6, 4))
plt.plot(energy_history, label='VQE Energy', color='cyan')
plt.axhline(y=-1.137, color='r', linestyle='--', label='Exact Energy (H2)')
plt.xlabel('Optimization Step')
plt.ylabel('Energy (Hartree)')
plt.title(f'VQE Convergence for {mol}')
plt.legend()
display(plt.gcf())
plt.close()

# Draw final circuit
final_qc = get_ansatz(result.x)
fig = circuit_drawer(final_qc, output='mpl')
display(fig)
plt.close(fig)

# Bloch sphere (qubit 0 of ground state)
sv = Statevector.from_instruction(final_qc)
rho = partial_trace(sv, [1])
a = np.sqrt(np.real(rho.data[0, 0]))
b_complex = rho.data[1, 0] / a if a > 1e-6 else 0
theta = 2 * np.arccos(np.clip(a, 0, 1))
phi = np.angle(b_complex) % (2 * np.pi)

try:
    fig3 = plot_bloch_multivector(sv)
    display(fig3)
    plt.close(fig3)
except Exception:
    pass
print(f"BLOCH_THETA={theta:.6f}")
print(f"BLOCH_PHI={phi:.6f}")
